<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🛡️ M1 作业：多 agent 自动代码审查

**一句话**：Senior Developer 查质量、Security Engineer 查安全（带 OWASP 搜索工具）、Tech Lead 综合两份结论决定批准 / 修改 / 升级人工。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ 作业 Homework</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 exercise.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：06 自动深度研究 Crew</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧾 评分：graded cells + unittests.py</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：团队在搭 CI 流水线，想把**初步代码审查**自动化以节省时间。很多代码问题有固定模式、可以自动发现和评估；另一些需要人类专家。
工具要能：分析代码改动 → 找出潜在问题 → **自主决定**是批准、提出修改，还是升级给人工审查。

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**评分规则（课程平台）**
- 只替换 `None`；可以加 cell 做实验，但评分器会忽略它们，答案只能写在给定 cell 里。
- **不要改变量名**（评分器按名字找答案），也不要改给定的配置。
- 带 `graded` tag 的 cell 是评分 cell，本 notebook 保留了它们的 metadata。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 第 0 步：拆问题

</div>

<img src="./images/agents-tasks-diagram.png" width="700">

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">👨‍💻 Senior Developer → Quality Analysis</div><div style="font-size:0.92em; line-height:1.6;">查风格问题、找潜在 bug、判断哪些是<b>必须修</b>（critical）哪些是<b>建议</b>（minor）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #f87171; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔐 Security Engineer → Security Review</div><div style="font-size:0.92em; line-height:1.6;">🔧 带工具。查漏洞、定风险等级、判断是否<b>阻塞</b>合并。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧭 Tech Lead → Review Decision</div><div style="font-size:0.92em; line-height:1.6;">读前两份结论：做最终决定、列出必改项、说明理由（批准 / 修改 / 升级人工）。</div></div></div></div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 1 步：导入与环境

</div>

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

课程平台已装好依赖；本地课程原写法是 `!pip install crewai[tools]==1.3.0`。本 notebook 用仓库的 `lc-foundations` kernel（crewai 1.15.22 + crewai_tools），无需安装。
`dill` 用来把结果序列化成 `results.dill` 交给评分；`unittests` 是本文件夹的本地测试（依赖 `dlai_grader`）。

</div>

In [1]:
from crewai import Agent, Task, Crew
import dill   # 比 pickle 能序列化更多对象，用于保存结果
import unittests   # 本文件夹的 unittests.py

In [2]:
import os
os.environ["CREWAI_TESTING"] = "true"   # 跳过首次运行的 tracing 交互询问
from utils import get_openai_api_key

# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set up the OpenAI model to use
os.environ["MODEL"] = 'gpt-4o-mini'   # 所有 agent 都没写 llm=，统一读这个默认模型

看一眼要审查的 PR：`app/user_auth.py` 的 diff。

In [3]:
# read and save the policies content
with open('code_changes.txt', 'r') as file:
    code_changes = file.read()
print(code_changes)

diff --git a/app/user_auth.py b/app/user_auth.py
index 8f23c4d..b9e7f2a 100644
--- a/app/user_auth.py
+++ b/app/user_auth.py
@@ -1,7 +1,32 @@
+from datetime import datetime
+import time
+
 def authenticate_user(username, password):
+    # Check if username or password is empty
+    if not username or not password:
+        return False
+    
+    # Query the database for the user
     user = db.query(f"SELECT * FROM users WHERE username = '{username}'")
+    
+    # Verify the user exists and password matches
     if user and user.password == password:
+        # Set session variables
         session['user_id'] = user.id
+        session['login_time'] = datetime.now()
+        
+        # Update last login timestamp
+        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")
+        
+        print(f"User {username} logged in successfully")
         return True
-    return False
+    else:
+        # Sleep to prevent timing attacks
+        time.sleep(1)
+       

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**预先看出的问题**（方便对照 agent 的结论）：
- 🔴 两处 **SQL 注入**：`f"SELECT ... '{username}'"`、`f"UPDATE ... {user.id}"`；
- 🔴 **明文比较密码** `user.password == password`（应存哈希、用常量时间比较）；
- 🟠 `print` 打出用户名到日志；只在失败分支 `sleep(1)` 反而**制造**了时间差，并不能防 timing attack；
- 🟡 `datetime` / `time` 导入、`logout_user` 无鉴权检查等小问题。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 2 步：工具实例（Exercise 1）

</div>

给 **Security Engineer** 两个工具：用 `SerperDevTool`（serper.dev 的 Google 搜索 API，类似 06 的 Exa）在 OWASP 网站搜相关安全规范，再用 `ScrapeWebsiteTool` 读每个找到的网页全文。OWASP 是致力于提升软件安全的非营利基金会。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异（crewai_tools 1.15.22，已实测）**
- **`base_url=os.getenv("DLAI_SERPER_BASE_URL")` 本地会直接报错**：课程平台才有这个代理变量；本地得到 `None`，而新版 `base_url: str` 不接受 `None` → `ValidationError: base_url Input should be a valid string`。最小修复：给 `os.getenv` 加默认值 `"https://google.serper.dev"`（即该字段原本的默认值），平台上行为不变。
- **`search_url="https://owasp.org"` 会被静默忽略**：`SerperDevTool` 根本没有这个字段（只有 `base_url`、`n_results`、`search_type`、`country` 等），搜索**并不会**限定在 OWASP。真要限定，得在搜索词里写 `site:owasp.org`（例如写进 task description 让 agent 照做）。

</div>

In [4]:
# GRADED CELL: Exercise 1
# import the required tools
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
# get the Serper API key
from utils import get_serper_api_key
serper_api_key = get_serper_api_key()   # SerperDevTool 实际从环境变量 SERPER_API_KEY 读 key

### START CODE HERE ###
# create the instance of the SerperDevTool. Set the search_url to "https://owasp.org"
serper_search_tool = SerperDevTool(
    search_url="https://owasp.org",   # ⚠️ 1.15.22 中无此字段，被忽略（见上方 🆕）
    base_url=os.getenv("DLAI_SERPER_BASE_URL", "https://google.serper.dev")   # 🆕 加默认值：本地没有课程代理变量
)

# create the instance of the ScrapeWebsiteTool, which does not need any arguments
scrape_website_tool = ScrapeWebsiteTool()
### END CODE HERE ###

测试会真的调用两个工具各一次（搜 `"SQL issues"`、抓 `https://owasp.org`），能跑通就算过。

In [5]:
# test the tools
unittests.test_tools(serper_search_tool, scrape_website_tool)

 You have 1 failed tests:

Failed test case: Couldn't run SerperSearchTool instance. 
Expected:
No exception,
but got:
Tool 'Search the internet with Serper' arguments validation failed: 1 validation error for SerperDevToolSchema
search_query
  Field required [type=missing, input_value={'query': 'SQL issues'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
Expected arguments: {"search_query": {"description": "Mandatory search query you want to use to search the internet", "title": "Search Query", "type": "string"}}
Required: ["search_query"].




<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10：这里会显示 `1 failed tests`，但工具本身没问题**。`unittests.test_tools` 调的是 `serper_search_tool.run(query="SQL issues")`；crewai 1.15.22 的 `BaseTool.run()` 会先按 `args_schema` 校验参数，而 `SerperDevToolSchema` 只认 `search_query` → `Field required`。旧版不校验所以能过。
下面用正确的参数名再调一次证明工具可用，顺便演示用 `site:owasp.org` 才能真正限定在 OWASP。

</div>

In [6]:
# 🧪 额外 cell（不计分）：用 schema 要求的参数名 search_query 调用
res = serper_search_tool.run(search_query="SQL injection site:owasp.org")
for r in res["organic"][:3]:
    print(r["link"])   # 全部来自 owasp.org

https://community.owasp.org/attacks/SQL_Injection
https://cheatsheetseries.owasp.org/cheatsheets/SQL_Injection_Prevention_Cheat_Sheet.html
https://community.owasp.org/attacks/SQL_Injection_Bypassing_WAF


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 3 步：三个 agent（Exercise 2–4）

</div>

每个 agent 定义 `role` / `goal` / `backstory` / `verbose`；Security Engineer 额外加 `tools`。课程对每个 agent 的要求：

| Agent | `role` 体现 | `goal` 聚焦 | `backstory` 强调 | 要让它明白 |
|---|---|---|---|---|
| Senior Developer | 代码质量分析专长 | 评估改动、决定哪些问题必须修 | 对质量问题做决策 | 区分 critical vs. minor |
| Security Engineer | 代码安全评估专长 | 找漏洞、定风险等级 | 对安全问题做决策 | 判断安全问题的严重程度 |
| Tech Lead | 管理代码审查流程 | 决定改动的审批路径 | 对审查流程做决策 | 最终拍板：批准还是升级 |

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**你的答案能过测试，但偏简短**（测试只检查字段是字符串、工具已挂上等）。想让结果更好：把上表「要让它明白」那一列直接写进 `goal` / `backstory`，例如 Senior Developer 的 goal 写成 *"Evaluate code changes and decide which issues are critical (must fix) vs minor"*。`seniro` 是拼写错误，不影响运行。

</div>

In [7]:
# GRADED CELL: Exercise 2

### START CODE HERE ### 

# Create the senior developer agent
senior_developer = Agent(
    role="Senior Developer",   # 你的答案
    goal="Review code",
    backstory=(
        "A seniro full stack developer who meticulously reviews code."
    ),
            
    # set verbose (suggested: True)
    verbose=True,
)

### END CODE HERE ###

In [8]:
# test the senior developer agent
unittests.test_senior_developer_agent(senior_developer)

 All tests passed!



In [9]:
# GRADED CELL: Exercise 3

### START CODE HERE ###

# Create the security engineer agent
security_engineer = Agent(
    role="Security Engineer",
    goal="Check security issues within code changes",
    backstory=( 
        "An experienced security engineer that specializes in security"
    ),
            
    # set verbose (suggested: True)
    verbose=True,
    # assign the tools you created in Exercise 1
    tools=[serper_search_tool, scrape_website_tool],   # 只有安全工程师需要上网查 OWASP
)

### END CODE HERE ###

In [10]:
# test the security engineer agent
unittests.test_security_engineer_agent(security_engineer)

 All tests passed!



In [11]:
# GRADED CELL: Exercise 4

### START CODE HERE ###

# Create the tech lead agent
tech_lead = Agent(
    role="Tech Lead",
    goal="Check code quality issues about code",
    backstory=(
            "You are a tech lead who knows everything about programming"
        ),
        
    # set verbose (suggested: True)
    verbose=True,
)
### END CODE HERE ###

In [12]:
# test the tech lead agent
unittests.test_tech_lead_agent(tech_lead)

 All tests passed!



<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 4 步：三个 task（Exercise 5–7）

</div>

通用参数：`description`（任务说明）、`expected_output`（格式和内容）、`agent`（谁做）、`context`（可选：哪些 task 的输出作为本 task 的上下文，可以多个）。本作业只有**最后一个** task 需要设 `context`。

| Task | `description` 要包含 | `expected_output` | agent |
|---|---|---|---|
| Quality Analysis | 审查代码、找 bug/问题、判断 critical 还是 minor；用 `{code_changes}`（**不要** f-string） | JSON，key 恰好是 `critical_issues` / `minor_issues` / `reasoning` | Senior Developer |
| Security Review | 查漏洞、识别安全问题、定风险等级、判断是否阻塞；用 `{code_changes}` | JSON，key：`security_vulnerabilities` / `blocking` / `highest_risk` / `security_recommendations` | Security Engineer |
| Review Decision | 判断能否批准、决定下一步、解释决定；用 `{code_changes}` | 简短报告：最终决定、必改项、批准意见、升级理由、额外建议 | Tech Lead，`context=[前两个 task]` |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**`name=` 不要改**：评分器靠 task 的 `name`（`"Analyze Code Quality"` / `"Review Security"` / `"Review Decision"`）找到它们。
**`{code_changes}` 不能用 f-string**：否则定义时就被替换（或 NameError），CrewAI 在 `kickoff(inputs=...)` 时才做插值。

</div>

In [13]:
# GRADED CELL: Exercise 5

### START CODE HERE ###

# Create the quality analysis task
analyze_code_quality = Task(
    description=( 
        "Review the following code: {code_changes}"
    ),
    
    expected_output=(
        "Find the code quality issues about code changes"   # ⚠️ 末尾无空格，会和下一行拼成 "changesThe"（不影响运行）
        "The output should be in the following json format"
        """
        {
            "critical_issues": "array of issues that must be fixed"
            "minor_issues": "array of suggested improvements",
            "reasoning": "explanation of decisions"
        }
        """
    ),
    
    name="Analyze Code Quality", #DO NOT CHANGE THIS NAME   # 评分器按 name 查找
    agent=senior_developer
)

### END CODE HERE ###

In [14]:
# test the quality analysis task
unittests.test_analyze_code_quality_task(analyze_code_quality)

 All tests passed!



你在 description 里额外告诉 agent **怎么用工具**：先用 SerperDevTool 找 OWASP 最佳实践，再把 URL 交给 ScrapeWebsiteTool 读全文——工具挂上了，agent 也知道何时用。

In [15]:
# GRADED CELL: Exercise 6

### START CODE HERE ###

# Create the security review task
review_security = Task(
    description=( 
        "Review possible security issues with {code_changes}"   # ⚠️ 相邻字符串直接拼接，段尾无空格会粘连
        # Add instructions for tool use to the task
        "Use the SerperDevTool to find the most relevant security best practices from OWASP"
        "and pass the URLs to the ScrapeWebsiteTool to get detailed information."
    ),
    
    expected_output=( 
        "The expected out should be in the following json format:"
        """
        {
            "security_vulnerabilities": "array of identified issues with risk levels"
            "blocking": "boolean indicating if security issues should block approval"
            "highest_risk": "the most severe risk level found"
            "security_recommendations": "specific fixes for vulnerabilities"
        }
        """
    ),
    
    agent=security_engineer,
    name="Review Security", # DO NOT CHANGE THIS NAME
)

### END CODE HERE ###

In [16]:
# test the review security task
unittests.test_review_security_task(review_security)

 All tests passed!



**`context=[analyze_code_quality, review_security]`**：显式指定后，Tech Lead 只拿到这两个 task 的输出（本例恰好就是前面全部 task，效果和默认相同，但意图更清楚）。

In [17]:
# GRADED CELL: Exercise 7

### START CODE HERE ###

# Create the review decision task
make_review_decision = Task(
    description=( 
        "Read the code changes from the provided {code_changes}. "
        "Then make a final decision wether to approve, reject, escaclate"
    ),
   
    expected_output=(
        "The output should be in the following json format:"
        """
        {
            "short_report": "a short report that includes the final decision",
            "required_changes": "The required suggested changes",
            "approval_comments": "If choosing to approve the code chnages, provide a comment to explain why.",
            "escalation_resoning": "If choosing to esclate the issues found within the code changes, provide the reasoning for it",
            "additional_recommendations": "Additional recommendations for the changed code"
        }
        """
    ), 
    
    agent=tech_lead,
    # add the two previous tasks as context
    context=[analyze_code_quality, review_security],    # 显式上下文：两份审查结论都拼进 prompt
    name="Review Decision", #DO NOT CHANGE THIS NAME
)

### END CODE HERE ###

In [18]:
# test the review decision task
unittests.test_make_review_decision_task(make_review_decision)

 All tests passed!



<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 5 步：组 Crew 并运行（Exercise 8–9）

</div>

课程说要传 `agents`、`tasks` 和 `llm`；实际上所有 agent 都从环境变量 `MODEL` 拿默认模型，`Crew` 只需要 `agents` + `tasks`。

In [19]:
# GRADED CELL: Exercise 8

### START CODE HERE ###

# Create the code review crew
crew = Crew(
    # add the list of agents
    agents=[senior_developer,security_engineer,tech_lead],
    # add the list of tasks    
    tasks=[analyze_code_quality, review_security, make_review_decision],   # 顺序：质量 → 安全 → 决策
)

### END CODE HERE ###

In [20]:
# test the crew
unittests.test_crew(crew)

 All tests passed!



`inputs` 字典的 key 必须和占位符同名：`{code_changes}` ← `inputs["code_changes"]`。

In [21]:
# define the inputs dictionary for the crew
inputs = {
    "code_changes": code_changes,   # 填进三个 task description 里的 {code_changes}
}

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异（crewai 1.15.22，已实测）**：原答案 `crew.kickoff(inputs=inputs)` 在 Jupyter 里会报 `RuntimeError: Agent execution was invoked synchronously from within a running event loop`；改用 `await crew.kickoff_async(inputs=inputs)`（内部 `asyncio.to_thread` 在工作线程里跑同步 kickoff），返回同样的 `CrewOutput`，后面的 `dill.dump` 不受影响。

</div>

In [22]:
# GRADED CELL: Exercise 9

### START CODE HERE ###

# kickoff the crew
result = await crew.kickoff_async(inputs=inputs)   # 原答案 crew.kickoff(...)，见上方 🆕

### END CODE HERE ###

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Task: Review the following code: diff --git a/app/user_auth.py b/app/user_auth.py                              │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                                                                                     │
│  +        # Sleep to prevent timing attacks                                                                     │
│  +        time.sleep(1)                                

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│      "critical_issues": [                                                                                       │
│          "SQL Injection Vulnerability: The code directly embeds user input in SQL queries, making it            │
│  vulnerable to SQL injection attacks.",                                                                         │
│          "Incorrect Password Handling: The authentication check compares plaintext passwords directly, which    │
│  is insecure. Passwords should be hashed and compared securely."                                                │
│      ],                                                                                                         │
│      "minor_issues": [                                                                                          │
│          "Unnecessary Sleep Function: The use of time.sleep(1) is a rudimentary approach to prevent timing      │
│  attacks and may result in a poor user experience. Consider implementing a more sophisticated rate limiting     │
│  mechanism.",                                                                                                   │
│          "Lack of Error Handling: There should be error handling for database queries (e.g., handling           │
│  connection issues or query failures).",                                                                        │
│          "Use of NOW() in SQL Update: While using NOW() in SQL works, it would be cleaner to use parameterized  │
│  queries or ORM methods instead to update timestamps.",                                                         │
│          "Logging Mechanism: Instead of using print statements for logging user activity, consider using a      │
│  logging framework to manage logs more effectively."                                                            │
│      ],                                                                                                         │
│      "reasoning": "The critical issues address security vulnerabilities that can lead to data breaches, making  │
│  it imperative to use prepared statements and secure password storage practices. Minor issues point to areas    │
│  for code improvement, enhancing maintainability, user experience, and robustness against potential issues."    │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Task: Review possible security issues with diff --git a/app/user_auth.py b/app/user_auth.py                    │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                                                                                     │
│  +        # Sleep to prevent timing attacks                                                                     │
│  +        time.sleep(1)                                

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'OWASP security best practices for SQL Injection prevention', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'SQL Injection Prevention - OWASP Cheat...
Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'OWASP security best practices for password handling', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Authentication - OWASP Cheat Sheet Series', '...
Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'OWASP security best practices for session management', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Session Management - OWASP Cheat Sheet Serie...


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│      "security_vulnerabilities": [                                                                              │
│          {                                                                                                      │
│              "issue": "SQL Injection Vulnerability",                                                            │
│              "risk_level": "critical"                                                                           │
│          },                                                                                                     │
│          {                                                                                                      │
│              "issue": "Incorrect Password Handling",                                                            │
│              "risk_level": "critical"                                                                           │
│          },                                                                                                     │
│          {                                                                                                      │
│              "issue": "Unnecessary Sleep Function",                                                             │
│              "risk_level": "minor"                                                                              │
│          },                                                                                                     │
│          {                                                                                                      │
│              "issue": "Lack of Error Handling",                                                                 │
│              "risk_level": "minor"                                                                              │
│          },                                                                                                     │
│          {                                                                                                      │
│              "issue": "Use of NOW() in SQL Update",                                                             │
│              "risk_level": "minor"                                                                              │
│          },                                                                                                     │
│          {                                                                                                      │
│              "issue": "Logging Mechanism",                                                                      │
│              "risk_level": "minor"                                                                              │
│          }                                                                                                      │
│      ],                                                                                                         │
│      "blocking": true,                                                                                          │
│      "highest_risk": "critical",                       

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Task: Read the code changes from the provided diff --git a/app/user_auth.py b/app/user_auth.py                 │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                                                                                     │
│  +        # Sleep to prevent timing attacks                                                                     │
│  +        time.sleep(1)                                

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│      "short_report": "The code changes contain critical security vulnerabilities that must be addressed before  │
│  approval. Therefore, the changes are rejected.",                                                               │
│      "required_changes": [                                                                                      │
│          "Implement parameterized queries or ORM methods to prevent SQL injection attacks.",                    │
│          "Use secure password hashing mechanisms (like bcrypt or Argon2id) instead of plaintext password        │
│  comparison.",                                                                                                  │
│          "Replace the time.sleep(1) with a more sophisticated rate-limiting solution for failed login           │
│  attempts.",                                                                                                    │
│          "Add proper error handling for database queries to manage exceptions and failures safely.",            │
│          "Use parameterized queries or ORM for the timestamp update instead of embedding SQL functions          │
│  directly.",                                                                                                    │
│          "Incorporate a logging framework to replace direct print statements for better log management."        │
│      ],                                                                                                         │
│      "approval_comments": "Not applicable as the changes cannot be approved due to critical security issues.",  │
│      "escalation_resoning": "The critical issues present significant security vulnerabilities that could lead   │
│  to data breaches, making it essential to handle user authentication and database interactions correctly.       │
│  Addressing these is crucial for protecting user data and application integrity.",                              │
│      "additional_recommendations": "Consider conducting a security audit of the entire application to identify  │
│  and rectify other potential vulnerabilities. Furthermore, implementing code review processes and automated     │
│  security scanning tools would enhance overall code quality and security."                                      │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 6 步：看结果、保存、检查格式

</div>

最终报告是第 3 个 task（Review Decision）的输出：`result.tasks_output[2]`（等同于 `result.raw`）。检查要求的内容是否都在，不在就回去改 task 定义。

In [23]:
from IPython.display import Markdown
Markdown(result.tasks_output[2].raw) 

```json
{
    "short_report": "The code changes contain critical security vulnerabilities that must be addressed before approval. Therefore, the changes are rejected.",
    "required_changes": [
        "Implement parameterized queries or ORM methods to prevent SQL injection attacks.",
        "Use secure password hashing mechanisms (like bcrypt or Argon2id) instead of plaintext password comparison.",
        "Replace the time.sleep(1) with a more sophisticated rate-limiting solution for failed login attempts.",
        "Add proper error handling for database queries to manage exceptions and failures safely.",
        "Use parameterized queries or ORM for the timestamp update instead of embedding SQL functions directly.",
        "Incorporate a logging framework to replace direct print statements for better log management."
    ],
    "approval_comments": "Not applicable as the changes cannot be approved due to critical security issues.",
    "escalation_resoning": "The critical issues present significant security vulnerabilities that could lead to data breaches, making it essential to handle user authentication and database interactions correctly. Addressing these is crucial for protecting user data and application integrity.",
    "additional_recommendations": "Consider conducting a security audit of the entire application to identify and rectify other potential vulnerabilities. Furthermore, implementing code review processes and automated security scanning tools would enhance overall code quality and security."
}
```

把整个 `CrewOutput` 用 `dill` 存成 `results.dill`——课程平台评分要用这个文件。

In [24]:
with open("results.dill", "wb") as f:
    dill.dump(result, f)

再检查前两个 task 的输出是否是带指定 key 的 JSON 字典（`utils.get_dict_keys` 尝试 `json.loads`）。
**不按「能否解析成 JSON」评分**——LLM 并不总能输出合法 JSON（比如套了 ```json 代码块）；后面模块会学强制结构化输出的方法。

In [25]:
from utils import get_dict_keys

# check the result of the first task

# Get the raw output
raw_output = result.tasks_output[0].raw   # 第 1 个 task：Analyze Code Quality

# See if it can be parsed as a dictionary, and get the keys
get_dict_keys(raw_output)

  ❌ Cannot parse as JSON



**期望输出：**
```
✅ Can be parsed as JSON dictionary
Keys: ['critical_issues', 'minor_issues', 'reasoning']
```

In [26]:
# check the result of the first task

# Get the raw output
raw_output = result.tasks_output[1].raw   # 第 2 个 task：Review Security（原注释写成了 first task）

# See if it can be parsed as a dictionary, and get the keys
get_dict_keys(raw_output)

  ❌ Cannot parse as JSON



**期望输出：**
```
✅ Can be parsed as JSON dictionary
Keys: ['security_vulnerabilities', 'blocking', 'highest_risk', 'security_recommendations']
```
做完可以换 agent / task 的定义多试几次，或拿自己的 commit 来审。

如果上面显示 `❌ Cannot parse as JSON`，多半是 gpt-4o-mini 把 JSON 包进了 ```` ```json ```` 代码块（之前一次运行就是这样，每次运行结果不同）。剥掉外壳再检查，key 通常都对：

In [27]:
# 🧪 额外 cell（不计分）：剥掉 ```json ... ``` 外壳再检查
for i in range(2):
    raw = result.tasks_output[i].raw.strip()
    print(repr(raw[:8]))   # 看开头是不是 ```json
    get_dict_keys(raw.removeprefix("```json").removesuffix("```").strip())

'```json\n'
  ✅ Can be parsed as JSON dictionary
  Keys: ['critical_issues', 'minor_issues', 'reasoning']

'```json\n'
  ✅ Can be parsed as JSON dictionary
  Keys: ['security_vulnerabilities', 'blocking', 'highest_risk', 'security_recommendations']



<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| 本地跑 `SerperDevTool(base_url=os.getenv("DLAI_SERPER_BASE_URL"))` | `ValidationError: base_url Input should be a valid string` | 本地无该变量 → 给 `os.getenv` 加默认值 `https://google.serper.dev` |
| 以为 `search_url=` 能把搜索限定在 OWASP | 搜到全网结果 | 1.15.22 无此字段、被忽略；在搜索词里用 `site:owasp.org` |
| Jupyter 里 `crew.kickoff()` | `RuntimeError ... running event loop` | `await crew.kickoff_async(...)` |
| 改了 task 的 `name` 或变量名 | 评分器找不到答案 | 按注释保持原名 |
| description 用 f-string | `{code_changes}` 提前被替换 / NameError | 普通字符串 + `kickoff(inputs=...)` |
| 期望 `expected_output` 保证 JSON | `❌ Cannot parse as JSON` | 只是提示；强制结构用 `output_pydantic` / `output_json` |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. 为什么只有 Security Engineer 配工具？</b></summary>

它需要去 OWASP 查最新的安全规范；质量审查和最终决策只需读代码和前面的结论。

</details>

<details><summary><b>2. Review Decision 为什么要设 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">context</code>？不设会怎样？</b></summary>

它要同时读质量和安全两份结论。本例不设也会默认拿到前面所有 task 的输出，但显式写出更清楚，也避免以后插入其它 task 时混入无关上下文。

</details>

<details><summary><b>3. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">{code_changes}</code> 在哪一步被替换成真实 diff？</b></summary>

`crew.kickoff(_async)(inputs={"code_changes": ...})` 时，CrewAI 对每个 task 的 description / expected_output 做模板插值。

</details>

<details><summary><b>4. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">result.tasks_output[2].raw</code> 和 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">result.raw</code> 的关系？</b></summary>

相同：`result.raw` 就是最后一个 task 的输出。

</details>

<details><summary><b>5. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">search_url=&quot;https://owasp.org&quot;</code> 在 1.15.22 里起作用吗？</b></summary>

不起作用，`SerperDevTool` 没有这个字段，被静默忽略；要限定站点需在查询里写 `site:owasp.org`。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：**按「质量 / 安全 / 决策」拆成三个专职 agent，用 `context` 把两份审查结论交给 Tech Lead**；工具只给需要上网的 agent，JSON 格式靠 `expected_output` 只是「请求」不是保证。

</div>